# Calculations and demos of the perturbation theory

- Bernardeau et al (https://arxiv.org/pdf/astro-ph/0112551)
- MUSIC validation
- MUSIC-like whitenoise generation

In [ ]:
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import camb
from colossus.cosmology import cosmology

In [ ]:
import logging

log = logging.getLogger(__name__)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

### Auxiliary functions

In [ ]:
seed = 137

In [ ]:
def generate_normal(size, *, loc=0, scale=1, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.normal(loc=loc, scale=scale, size=size)

def generate_integer(size, *, low=0, high=10, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.integers(low=low, high=high, size=size)

def generate_uniform(size, *, low=0, high=1, seed=137):
    rng = np.random.default_rng(seed=seed)
    return rng.uniform(low=low, high=high, size=size)

In [ ]:
from scipy.interpolate import RegularGridInterpolator

In [ ]:
def interpolate_field(x, field, dk, method='linear'):
    r'''
    Interpolate a grid-based field onto particle positions using periodic
    boundaries.

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Particle positions in the simulation box.
    field : ndarray
        The grid-based field (e.g. a displacement field) defined on a
        regular grid.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    method : str, optional
        The interpolation method to use. This can be 'linear', 'nearest',
        or 'cubic'. The default is 'linear'.

    Returns
    -------
    interp_values : ndarray of shape (N,)
        Field values interpolated at the particle positions.
    '''
    nvox = field.shape
    mesh = (np.arange(dk/2, n*dk+dk/2, dk) for n in nvox)
    interpolator = RegularGridInterpolator(
        mesh,
        field,
        method=method,
        bounds_error=False,
        fill_value=None  # Extrapolate using periodic wrapping if needed
    )
    return interpolator(x)

In [ ]:
def compute_overdensity(density_field):
    r'''
    Compute the overdensity field defined as
    .. math::
        $\delta(\mathbf{x}) = \frac{\rho(\mathbf{x}) - \bar{\rho}}{\bar{\rho}}$.

    where :math:`\rho(\mathbf{x})` is the density field and :math:`\bar{\rho}`
    is the mean density.

    Parameters
    ----------
    density_field : ndarray
        The density field.

    Returns
    -------
    overdensity_field : ndarray
        The overdensity field :math:`\delta(\mathbf{x})`.
    '''
    mean_density = np.mean(density_field)
    return (density_field - mean_density) / mean_density

### Cosmology

In [ ]:
from colossus.cosmology import cosmology

In [ ]:
def hubble_a(a, H0, omega_m, omega_l):
    r'''
    Computes the Hubble parameter :math:`H(a)` at scale factor :math:`a`.

    The Hubble parameter is given by

    .. math::
        H(a) = H_0\,\sqrt{\Omega_m\,a^3 + (1 - \Omega_m - \Omega_\Lambda)\,a^2 + \Omega_\Lambda},

    where :math:`a` is the scale factor normalized to 1 at present. :math:`H_0`
    is the Hubble constant, :math:`\Omega_m` is the present-day matter
    density parameter and :math:`\Omega_\Lambda` is the present-day dark
    energy density parameter.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    H0 : float
        Hubble constant in km/s/Mpc.
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The Hubble parameter :math:`H(a)`.
    '''
    return H0 * np.sqrt(omega_m / a**3 + (1 - omega_m - omega_l) / a**2 + omega_l)

In [ ]:
def F_omega(a, omega_m, omega_l):
    r'''
    Computes the linear growth rate factor for first-order Lagrangian
    perturbation.

    This function returns the factor :math:`F_\omega(a)`, defined by

    .. math::
        F_\omega(a) = \left[\Omega(a)\right]^{0.6},

    where the effective matter density parameter :math:`\Omega(a)` is computed
    as

    .. math::
        \Omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F_\omega` approximates the logarithmic derivative of the linear
    growth factor :math:`D_1` with respect to the scale factor :math:`a`, i.e.

    .. math::
        f \equiv \frac{d\ln(D_1)}{d\ln(a)}.


    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The linear growth rate :math:`F_\omega(a)`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return np.power(omega_a, 5.0/9.0)  # Bernardeau et al. 2001, eq. 101a

In [ ]:
def F2_omega(a, omega_m, omega_l):
    r'''
    Computes the second-order growth rate factor for second-order
    Lagrangian perturbation theory corrections.

    This function returns the factor :math:`F2_\omega(a)`, defined by

    .. math::
        F2_\omega(a) = 2\,\left[\Omega(a)\right]^{\frac{4}{7}},

    where the effective matter density parameter :math:`\Omega(a)` is computed
    as

    .. math::
        \Omega(a) = \frac{\omega_m}{\omega_m + a\,(1 - \omega_m - \omega_l) + \omega_l\,a^3}.

    :math:`F2_\omega` is used in second-order Lagrangian perturbation theory
    to scale the second-order displacement field and its time derivative,
    thereby accounting for non-linear corrections to the growth of structure.

    Parameters
    ----------
    a : float
        Scale factor (normalized to 1 at present).
    omega_m : float
        Present-day matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.

    Returns
    -------
    float
        The second-order growth rate :math:`F2_\omega(a)`.
    '''
    omega_a = omega_m / (omega_m + a * (1 - omega_m - omega_l) + a**3 * omega_l)
    return 2 * np.power(omega_a, 6.0/11.0)  # Bernardeau et al. 2001, eq. 101b

In [ ]:
def D1_z(z, H0, omega_m, omega_b, omega_l, sigma8, ns, de_model, de_params):
    '''
    Calculate the linear growth factor :math:`D_1 (z)` normalized to 1
    at :math:`a = 1`.

    For :math:`w_0` and CPL dark energy, we use the Colossus
    implementation of Eq. (11) from Linder & Jenkins (2003).
    See paper at https://arxiv.org/pdf/astro-ph/0305286.pdf.

    Parameters:
    -----------
    z : float
        Redshift at which the linear growth factor is calculated.
    H0 : float
        Hubble constant in km/s/Mpc.
    omega_m : float
        Present-day matter density parameter.
    omega_b : float
        Present-day baryonic matter density parameter.
    omega_l : float
        Present-day dark energy density parameter.
    sigma8 : float
        RMS matter fluctuation amplitude at 8 Mpc/h.
    ns : float
        Scalar spectral index of the primordial power spectrum.
    de_model : str
        Dark energy model. Options are 'Lambda', 'w0', 'CPL'.
    de_params : list or array-like
        Dark energy model parameters. For 'Lambda' and 'w0', it is a
        single element list containing the dark energy equation of state
        parameter :math:`w`. For 'CPL', it is a two element list
        containing :math:`w` and :math:`w_a`.

    Returns:
    --------
    D1 : float
        The linear growth factor :math:`D_1(z)` normalized to 1 at
        :math:`a = 1`.
    '''
    # Calculating the curvature
    omega_k = 1.0 - omega_m - omega_l
    if np.abs(omega_k) <= 1e-5:
        flat = True
        log.info('Flat cosmology.')
    else:
        flat = False
        log.info(f'Non-flat cosmology; {omega_k = }, {omega_m = :.4f}, {omega_l = :.4f}')
    # Zero CMB temperature can cause issues in colossus. This small value
    # should not cause any significant errors at late times in relevant
    # cosmologies. TODO: Implement a non-zero `omega_r`
    T_cmb = 0.001
    
    params = {  # Common cosmological parameters for all models
        'flat': flat, 'H0': H0, 'Om0': omega_m, 'Ob0': omega_b,
        'sigma8': sigma8, 'ns': ns, 'Tcmb0': T_cmb
    }
    if de_model == 'Lambda':
        if not flat:
            params.update({'Ode0': omega_l})
        cosmo = cosmology.setCosmology('LCDM', **params)
    elif de_model == 'w0':
        params.update({'de_model': 'w0', 'w0': de_params[0]})
        if not flat:
            params.update({'Ode0': omega_l})
        cosmo = cosmology.setCosmology('wCDM', **params)
    elif de_model == 'CPL':
        params.update({'de_model': 'w0wa', 'w0': de_params[0], 'wa': de_params[1]})
        if not flat:
            params.update({'Ode0': omega_l})
        cosmo = cosmology.setCosmology('w0waCDM', **params)
    else:
        raise ValueError('Invalid dark energy model. Options are "Lambda", "w0", "CPL".')
    D1 = cosmo.growthFactorUnnormalized(z) / cosmo.growthFactorUnnormalized(0.0)
    log.info(f'Normalized linear growth factor D1(z={z:.2f})/D1(z=0) = {D1:.2e}')
    return D1

### Power spectrum

In [ ]:
def init_camb_cosmology(
        H0=67.4, ombh2=0.0224, omch2=0.120, omega_k=0.0,
        de_model='Lambda', de_params=None,
        nonlinear=False, halofit_version='mead2020'):
    '''
    Initialize a CAMB cosmology object with specified parameters. The
    default cosmological parameters are from the Planck 2018 results.
    
    Parameters:
    -----------
    H0 : float
        Hubble constant in km/s/Mpc.
    ombh2 : float
        Physical baryon density parameter.
    omch2 : float
        Physical cold dark matter density parameter.
    omega_k : float
        Curvature density parameter.
    kmax : float
        Maximum wavenumber in h/Mpc.
    de_model : str
        Dark energy model. Options are 'Lambda', 'w0', 'CPL'.
    de_params : list or array-like
        Dark energy model parameters. For 'Lambda' and 'w0', it is a
        single element list containing the dark energy equation of state
        parameter $w$. For 'CPL', it is a two element list containing
        $w$ and $w_a$.
    nonlinear : bool
        If True, include non-linear corrections using Halofit.
    halofit_version : str
        Version of the Halofit model to use for non-linear corrections.
        Check ``camb.nonlinear.Halofit`` for available models.
    '''
    camb_params = camb.CAMBparams()
    camb_params.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, omk=omega_k)
    if de_model == 'w0':
        if not (isinstance(de_params, (list, tuple)) and len(de_params) == 1):
            raise ValueError("`de_params` must be a 1-element list for 'w0'.")
        camb_params.DarkEnergy = camb.dark_energy.DarkEnergyFluid()
        camb_params.DarkEnergy.set_params(w=de_params[0])
    elif de_model == 'CPL':
        if not (isinstance(de_params, (list, tuple)) and len(de_params) == 2):
            raise ValueError("`de_params` must be a 2-element list for 'CPL'.")
        camb_params.DarkEnergy = camb.dark_energy.DarkEnergyFluid()
        camb_params.DarkEnergy.set_params(w=de_params[0], wa=de_params[1])
    
    if nonlinear:
        log.info(f'Using non-linear corrections with Halofit model `{halofit_version}`')
        camb_params.NonLinear = camb.model.NonLinear_both
        camb_params.NonLinearModel = camb.nonlinear.Halofit()
        camb_params.NonLinearModel.set_params(halofit_version=halofit_version)
    else:
        log.info('Using linear theory only.')
        camb_params.NonLinear = camb.model.NonLinear_none

    return camb_params

In [ ]:
def calculate_sigma8(camb_params, *, z=0, As=2.097e-09, ns=0.965, kmax=1.0):
    r'''
    Calculate the RMS matter fluctuation amplitude $\sigma_8$ at given
    redshift $z$.
    
    This function uses the CAMB package to compute the matter power
    spectrum and extract the $\sigma_8$ value. The default cosmological
    parameters are from the Planck 2018 results.

    Parameters:
    -----------
    camb_params : camb.CAMBparams
        CAMB parameters object initialized with cosmological parameters.
    z : float; default=0
        Target redshift.
    As : float
        Comoving curvature power at $k = 0.05\,\mathrm{Mpc}^{-1}$.
        This is the amplitude of the primordial power spectrum at large
        scales, typically set to match the observed $\sigma_8$.
    ns : float
        Scalar spectral index.
    kmax : float
        Maximum wavenumber in $h^{-1}\,\mathrm{Mpc}$.

    Returns:
    --------
    float
        The RMS matter fluctuation amplitude $\sigma_8$ at redshift $z$.
    '''
    camb_params.InitPower.set_params(As=As, ns=ns)
    camb_params.set_matter_power(redshifts=[z], kmax=kmax)
    results = camb.get_results(camb_params)
    sigma8 = results.get_sigma8()[0]
    
    log.info(f'RMS matter fluctuation amplitude {sigma8 = :.4f} (from {As = :.3e})')
    return sigma8

In [ ]:
def camb_spectrum(
        camb_params, *, z=127, As=2.097e-09, ns=0.965, kmin=0.01, kmax=1.0,
        npoints=1024, sigma8_init: float = None):
    r'''
    Calculate the matter power spectrum using CAMB. The default
    cosmological parameters are from the Planck 2018 results.

    Parameters:
    -----------
    z : float or list of float
        Redshifts at which the linear power spectrum is calculated.
    As : float
        Scalar amplitude of the primordial power spectrum.
    ns : float
        Scalar spectral index.
    kmin : float
        Minimum wavenumber in h/Mpc.
    kmax : float
        Maximum wavenumber in h/Mpc.
    npoints : int
        Number of wavenumber points.
    sigma8_init : float
        Rescale the matter power spectrum to the given $\sigma_8$ value.
    '''
    # Optional rescaling of the `As` amplitude to match a desired sigma8
    if sigma8_init is not None:
        sigma8 = calculate_sigma8(camb_params, z=0, As=As, ns=ns, kmax=kmax)

        As *= (sigma8_init / sigma8)**2
        camb_params.InitPower.set_params(As=As, ns=ns)
        camb_params.set_matter_power(redshifts=[0], kmax=kmax)
        results = camb.get_results(camb_params)
        sigma8 = results.get_sigma8()[0]

        log.info(f'Rescaled matter fluctuation amplitude {sigma8 = :.4f} (from {As = :.3e})')

    # Calculating P(k) at redshift `z`
    camb_params.set_matter_power(redshifts=np.atleast_1d(z).tolist(), kmax=kmax)
    results = camb.get_results(camb_params)
    kh, _, pk = results.get_matter_power_spectrum(
                                minkh=kmin, maxkh=kmax, npoints=npoints)
    return kh, pk

## Fourier grid 

In [ ]:
def cubic_voxels(nmesh, Lbox):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters
    ----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.

    Returns
    -------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    ref_L = np.min(Lbox)
    nvox = np.ceil(Lbox / (ref_L / nmesh)).astype(int)
    nvox = (nvox + nvox % 2).astype(int)  # Ensure even number of voxels
    log.info('Mesh: Nx={}, Ny={}, Nz={}'.format(*nvox))
    dk = ref_L / nvox[Lbox.index(ref_L)]
    log.info(f'Step size: {dk}')
    return nvox, dk

In [ ]:
def fourier_grid(nvox, dk, hermitian=False):
    r'''
    Construct a 3D Fourier space grid.

    This function generates a three-dimensional array of wavevector
    components (``kvec``) and computes the corresponding magnitude
    (``kmod``) for a cubic grid with ``nmesh`` points per side within
    a box of size ``Lbox``.

    The grid is then constructed using the FFT frequencies:
    - For the first two dimensions, the full set of FFT frequencies is
      computed using ``np.fft.fftfreq``.
    - For the third dimension, if the input field is real-valued (i.e.
      if Hermitian symmetry is assumed), the reduced set of frequencies
      is computed using ``np.fft.rfftfreq``.
    
    When ``hermitian`` is True, the output grid reflects the storage
    scheme of a real FFT, and the shape of ``kvec`` is
    :math:`(3, {\rm nmesh}, {\rm nmesh}, {\rm nmesh}//2+1)`. Otherwise,
    a full grid with shape :math:`(3, {\rm nmesh}, {\rm nmesh}, {\rm nmesh})`
    is returned.

    Parameters
    ----------
    nvox : 
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    hermitian : bool, optional
        If True, assume the field has Hermitian symmetry (i.e., it is
        real-valued) and use the reduced FFT along the last dimension.
        The default is False.

    Returns
    -------
    kvec : ndarray
        A three-dimensional array of wavevector components with shape:
          - :math:`(3, {\rm nmesh}, {\rm nmesh}, {\rm nmesh}//2+1)` if
          ``hermitian`` is True.
          - :math:`(3, {\rm nmesh}, {\rm nmesh}, {\rm nmesh})` if
          ``hermitian`` is False.
        Each sub-array corresponds to the ``x``, ``y``, or ``z`` component
        of the wavevector.
    kmod : ndarray
        The magnitude of the wavevector at each grid point, computed as
        :math:`\|\mathbf{k}\| = \sqrt{k_x^2 + k_y^2 + k_z^2}`.
    '''
    kx = np.fft.fftfreq(nvox[0]) * 2 * np.pi / dk
    ky = np.fft.fftfreq(nvox[1]) * 2 * np.pi / dk
    if hermitian:
        kz = np.fft.rfftfreq(nvox[2]) * 2 * np.pi / dk
    else:
        kz = np.fft.fftfreq(nvox[2]) * 2 * np.pi / dk
    kvec = np.array(np.meshgrid(kx, ky, kz, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod

## 1LPT - Zel'dovich approximation

In [ ]:
def lpt1(x, field, nvox, dk, D1, dD1, h, counter=False):
    r'''
    Apply first-order Lagrangian Perturbation Theory (LPT), i.e., the
    Zel'dovich approximation, to generate perturbed particle positions
    and velocities.

    In this approximation, particles are displaced from their initial
    (Lagrangian) positions :math:`\mathbf{q}` to their final (Eulerian)
    positions :math:`\mathbf{x}` using a displacement field
    :math:`\mathbf{\Psi}`:

    .. math::
        \mathbf{x}(\mathbf{q}, t) = \mathbf{q} + D_1(t) \, \mathbf{\Psi}(\mathbf{q}),

    where :math:`D_1(t)` is the linear growth factor and :math:`\mathbf{\Psi}(\mathbf{q})`
    is the displacement field computed from the initial density
    perturbations. The displacement field is related to the gravitational
    potential, and in Fourier space, it is calculated from the
    overdensity field :math:`\delta(\mathbf{k})`:

    .. math::
        \mathbf{\Psi}(\mathbf{k}) =
            -i \, \frac{\mathbf{k}}{|\mathbf{k}|^2} \, \delta(\mathbf{k})
        \quad \text{for } |\mathbf{k}| > 0,

    where :math:`\mathbf{k}` is the wavevector.
    
    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Initial unperturbed particle positions (Lagrangian coordinates),
        in comoving Mpc/h.
    field : ndarray
        Real-space overdensity field :math:`\delta(\mathbf{x})` defined
        on a regular grid, where :math:`\mathbf{x}` are the comoving
        coordinates.
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    D1 : float
        The linear growth factor :math:`D_1(z)` at the desired redshift.
    dD1 : float
        A prefactor for the velocity calculation, typically related to the
        time derivative of the growth factor (e.g. :math:`\dot{D}_1` or
        :math:`H(a)f(a)` where f is the growth rate). The code uses this
        in a non-standard velocity formula.
    h : float
        The dimensionless Hubble parameter, :math:`h = H_0 / 100`, where
        :math:`H_0` is the Hubble constant.
    counter : bool
        If True, applies a global sign flip to the Fourier-space density
        field (equivalent to a :math:`\pi` phase shift). This is useful
        for running "counter-phased" simulations to reduce sample
        variance.
    
    Returns
    -------
    xpert : ndarray of shape (N, 3)
        Perturbed particle positions (Eulerian coordinates) in comoving
        Mpc/h. Positions are wrapped to lie within the periodic box.
    v : ndarray of shape (N, 3)
        Particle peculiar velocities in km/s. The velocity is computed
        according to the specific formula implemented in this function:

        .. math::
            \mathbf{v} = \frac{D_1(t) \cdot dD_1 \cdot \mathbf{\Psi}}{h}.

    Notes
    -----
    **Overview of the Implementation:**

    1.  **Fourier Grid Setup:** A 3D Fourier grid (``kvec``) is constructed
        for a mesh of size ``(Nx, Ny, Nz)``. For a real-valued field, a
        reduced FFT is used (Hermitian symmetry), so the k-space arrays
        have a shape of ``(Nx, Ny, Nz//2 + 1)``.

    2.  **Displacement Field Calculation:**
        - The Fourier-transform of the overdensity field is computed to
          get :math:`\delta(\mathbf{k})`.
        - The Fourier-space displacement field :math:`\mathbf{\Psi}(\mathbf{k})`
          is computed for each spatial component. Division by zero at the
          DC mode (:math:`|\mathbf{k}| = 0`) is avoided.
        - An inverse FFT converts :math:`\mathbf{\Psi}(\mathbf{k})` back
          to a real-space grid.

    3.  **Interpolation and Particle Update:**
        - The gridded displacement field :math:`\mathbf{\Psi}` is
          interpolated to the Lagrangian particle positions :math:`\mathbf{q}`.
        - Particle positions are updated to their Eulerian coordinates:

          .. math::
              \mathbf{x}_{\text{pert}} = \mathbf{q} + D_1(t) \, \mathbf{\Psi}(\mathbf{q})

        - Particle velocities are computed using the interpolated
          displacement field :math:`\mathbf{\Psi}`, the growth factor
          ``D1``, and the velocity prefactor ``dD1``. The final result
          is divided by ``h`` to obtain units of km/s.
    '''
    kvec, kmod = fourier_grid(nvox, dk, hermitian=True)
    delta_k = np.fft.rfftn(field)
    delta_k = delta_k * np.exp(1j * np.pi) if counter else delta_k
    mask = kmod > 0.0  # Avoid division by zero at k = 0
    xpert = np.zeros_like(x, dtype=np.float32)
    v = np.zeros_like(x, dtype=np.float32)
    for i, xi in enumerate(('x', 'y', 'z')):
        psi1_ki = np.zeros_like(kmod, dtype=complex)
        psi1_ki[mask] = -1j * kvec[i, mask] / (kmod[mask]**2) * delta_k[mask]
        disp_field = np.fft.irfftn(psi1_ki, s=nvox, axes=(0, 1, 2))
        max_disp = np.max(np.abs(disp_field))
        log.info(f"Maximal '{xi}' displacement: {max_disp*1e3:.3f} kpc/h; "
                 f"in units of mean particle separation: {max_disp / dk:.3f}")
        disp_field_interp = interpolate_field(x, disp_field, dk)
        xpert[:, i] = x[:, i] + D1 * disp_field_interp
        v[:, i] = disp_field_interp * D1 * dD1
    return xpert, v / h

In [ ]:
z = 63
a = 1.0 / (1.0 + z)
H0 = 67.74  # [km/s/Mpc]
omega_m = 0.3089
omega_b = 0.0486
omega_l = 0.6911
s8 = 0.811
ns = 0.9665

de_model = 'Lambda'
de_params = []  # No parameters needed for Lambda model
D1 = D1_z(
    z, H0, omega_m, omega_b, omega_l, s8, ns, de_model, de_params)
D2 = (3.0/7.0) * D1**2 * omega_m**(-1/143)  # Bernardeau et al. 2001, eq. 97
print(f'D1(z={z}) = {D1:.6f}, D2(z={z}) = {D2:.6f}')

dD1 = a * hubble_a(a, H0, omega_m, omega_l) * F_omega(a, omega_m, omega_l)
dD2 = a * hubble_a(a, H0, omega_m, omega_l) * F2_omega(a, omega_m, omega_l)
print(f'dD1(z={z}) = {dD1:.6f}, dD2(z={z}) = {dD2:.6f}')

In [ ]:
nmesh = 32
Lbox = [100, 100, 50]
periodic = [0, 0, 1]
dk, nvox = cubic_voxels(nmesh, Lbox)

field = generate_normal(size=nvox, seed=seed)
field = compute_overdensity(field)

x = generate_uniform((2000, 3), seed=seed) * np.array(Lbox)
x_pert, v_pert = lpt1(x, field, nvox, dk, D1, dD1, h=H0/100, counter=False)
x_pert = np.where(periodic, np.mod(x_pert, Lbox), x_pert)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

idx = [0, 1]

ax = axes[0]
ax.scatter(*x[:, idx].T, c='k', s=4**2, ec='none', alpha=0.5)
ax.set_title('Initial positions', loc='left', fontsize=10)

ax = axes[1]
ax.scatter(*x_pert[:, idx].T, c='tab:red', s=4**2, ec='none', alpha=0.5)
ax.set_title('Perturbed (LPT1) positions', loc='left', fontsize=10)

ax = axes[2]
disp = np.column_stack((x[:, idx].ravel(), x_pert[:, idx].ravel()))
ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
ax.set_title('Displacement field', loc='left', fontsize=10)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    disp = np.column_stack((x[:, idx].ravel(), x_pert[:, idx].ravel()))
    ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
x = generate_uniform((10000, 3), seed=seed) * np.array(Lbox)
x_pert, v_pert = lpt1(x, field, nvox, dk, D1, dD1, h=H0/100, counter=False)
x_pert = np.where(periodic, np.mod(x_pert, Lbox), x_pert)

In [ ]:
def histogram(x, bins=50):
    '''TODO'''
    hist, edge = np.histogram(x, bins=bins)
    bin_c = (edge[:-1] + edge[1:]) / 2
    bin_w = np.diff(edge)  # Width of each bin
    return bin_c, hist, bin_w

In [ ]:
nr, nc = 1, 4
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

ax = axes[0]

ax.set_title('Matter power spectrum of initial positions', loc='left', fontsize=10)

ax = axes[1]

ax.set_title('Matter power spectrum of perturbed positions', loc='left', fontsize=10)

ax = axes[2]
c, h, w = histogram(np.linalg.norm(x_pert - x, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='k', alpha=0.5)
ax.set_yscale('log')
ax.set_title('Distribution of particle displacements', loc='left', fontsize=10)

ax = axes[3]
c, h, w = histogram(np.linalg.norm(v_pert, axis=1), bins=100)
ax.bar(c, h, w, lw=0.5, color='k', alpha=0.5)
ax.set_title('Distribution of particle velocities', loc='left', fontsize=10)

plt.show()

## 2LPT

In [ ]:
def lpt2(x, field, nvox, dk, D1, dD1, D2, dD2, h, counter=False):
    r'''
    Apply second-order Lagrangian Perturbation Theory (2LPT) to generate
    perturbed particle positions and velocities.

    This method provides a more accurate description of particle
    trajectories than the Zel'dovich approximation (1LPT) by including
    the second-order term in the displacement. The final (Eulerian)
    position :math:`\mathbf{x}` is computed from the initial (Lagrangian)
    position :math:`\mathbf{q}` as:

    .. math::
        \mathbf{x}(\mathbf{q}, t) = \mathbf{q} - D_1(t)\mathbf{\Psi}^{(1)}(\mathbf{q}) + D_2(t)\mathbf{\Psi}^{(2)}(\mathbf{q})

    where :math:`\mathbf{\Psi}^{(1)}` and :math:`\mathbf{\Psi}^{(2)}`
    are the first- and second-order displacement fields, and :math:`D_1`
    and :math:`D_2` are the corresponding linear and second-order growth
    factors.

    The **first-order field** is calculated from the overdensity :math:`\delta`
    as in 1LPT:
    
    .. math::
        \mathbf{\Psi}^{(1)}(\mathbf{k}) = -i \frac{\mathbf{k}}{|\mathbf{k}|^2} \delta(\mathbf{k}).

    The **second-order field** is derived from a scalar potential :math:`\phi^{(2)}`,
    where :math:`\mathbf{\Psi}^{(2)} = -\nabla\phi^{(2)}`. The potential
    itself is sourced by a quadratic source term :math:`S(\mathbf{x})`
    from the spatial derivatives of the first-order displacement.
    Following standard 2LPT theory, one may compute

    .. math::
        S(\mathbf{x}) =
            \frac{\partial \Psi^{(1)}_x}{\partial x}\,\frac{\partial \Psi^{(1)}_y}{\partial y}
            + \frac{\partial \Psi^{(1)}_x}{\partial x}\,\frac{\partial \Psi^{(1)}_z}{\partial z}
            + \frac{\partial \Psi^{(1)}_y}{\partial y}\,\frac{\partial \Psi^{(1)}_z}{\partial z}
            - \left[
                \left(\frac{\partial \Psi^{(1)}_x}{\partial y}\right)^2
                + \left(\frac{\partial \Psi^{(1)}_x}{\partial z}\right)^2
                + \left(\frac{\partial \Psi^{(1)}_y}{\partial z}\right)^2
            \right].
         
    In the code we denote:
        - $dPxx = \frac{\partial \Psi^{(1)}_x}{\partial x}$,
        - $dPxy = \frac{\partial \Psi^{(1)}_x}{\partial y}$,
        - $dPxz = \frac{\partial \Psi^{(1)}_x}{\partial z}$,
        - $dPyy = \frac{\partial \Psi^{(1)}_y}{\partial y}$,
        - $dPyz = \frac{\partial \Psi^{(1)}_y}{\partial z}$,
        - $dPzz = \frac{\partial \Psi^{(1)}_z}{\partial z}$,
         
    and then set

    .. math::
        S(\mathbf{x}) =
            dPxx\,dPyy + dPxx\,dPzz + dPyy\,dPzz - (dPxy^2 + dPxz^2 + dPyz^2).
        
    The Poisson equation in Fourier space is solved for the second-order
    potential:

    .. math::
        \phi^{(2)}(\mathbf{k}) = -\frac{S(\mathbf{k})}{|\mathbf{k}|^2},
         
    with the $k=0$ mode appropriately masked. The second-order displacement
    in Fourier space is then given by

    .. math::
        \Psi^{(2)}_i(\mathbf{k}) = i\,k_i\,\phi^{(2)}(\mathbf{k}),
         
    and an inverse FFT yields the real-space second-order displacement
    field.
    

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Initial unperturbed particle positions (Lagrangian coordinates),
        in comoving Mpc/h.
    field : ndarray
        Real-space overdensity field :math:`\delta(\mathbf{x})` defined
        on a regular grid, where :math:`\mathbf{x}` are the comoving
        coordinates.
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    D1 : float
        The linear growth factor :math:`D_1(z)` at the desired redshift.
    dD1 : float
        A prefactor for the velocity calculation, typically related to the
        time derivative of the growth factor (e.g. :math:`\dot{D}_1` or
        :math:`H(a)f(a)` where f is the growth rate). The code uses this
        in a non-standard velocity formula.
    D2 : float
        The second-order growth factor :math:`D_2(z)` at the desired
        redshift.
    dD2 : float
        A prefactor for the second-order velocity term.
    h : float
        The dimensionless Hubble parameter, :math:`h = H_0 / 100`.
    counter : bool
        If True, applies a global sign flip to the Fourier-space density
        field (equivalent to a :math:`\pi` phase shift). This is useful
        for running "counter-phased" simulations to reduce sample
        variance.

    Returns
    -------
    xpert : ndarray of shape (N, 3)
        Perturbed particle positions (Eulerian coordinates) in comoving
        Mpc/h, wrapped within the periodic box.
    v : ndarray of shape (N, 3)
        Particle peculiar velocities in km/s, computed as:

        .. math::
            \mathbf{v} = \frac{-D_1 \cdot dD_1 \cdot \mathbf{\Psi}^{(1)} + D_2 \cdot dD_2 \cdot \mathbf{\Psi}^{(2)}}{h}.

    Notes
    -----
    **Overview of the Implementation:**

    1.  **First-Order Displacement:** The first-order displacement field,
        :math:`\mathbf{\Psi}^{(1)}`, is computed from the Fourier-space
        overdensity field, identical to the Zel'dovich approximation.

    2.  **Second-Order Source:** The spatial derivatives of the first-order
        field (i.e., the deformation tensor :math:`\partial \Psi_i^{(1)} / \partial q_j`)
        are calculated using FFTs. These are then combined to form the
        source term for the second-order potential.

    3.  **Second-Order Displacement:** The Poisson equation for the
        second-order potential :math:`\phi^{(2)}` is solved in Fourier
        space. The second-order displacement field, :math:`\mathbf{\Psi}^{(2)}`,
        is then found by taking the gradient of this potential, again in
        Fourier space.

    4.  **Interpolation and Particle Update:**
        - Both the first- and second-order displacement fields are
          interpolated from the grid to the Lagrangian particle positions.
        - Particle positions and velocities are updated by combining the
          interpolated first- and second-order contributions.

    .. warning::
        The update equations in this implementation use a specific sign
        convention (:math:`-D_1\mathbf{\Psi}^{(1)} + D_2\mathbf{\Psi}^{(2)}`).
        Ensure this is consistent with the definitions of the growth
        factors and displacement fields used in your analysis.
    '''
    kvec, kmod = fourier_grid(nvox, dk, hermitian=True)
    delta_k = np.fft.rfftn(field)
    delta_k = delta_k * np.exp(1j * np.pi) if counter else delta_k
    mask = kmod > 0.0  # Avoid division by zero at k = 0
    xpert = np.zeros_like(x, dtype=np.float32)
    v = np.zeros_like(x, dtype=np.float32)
    axes = (0, 1, 2)  # Axes for the FFT operations

    # ------------------------------
    # 1. First-order displacement (Psi^(1))
    # ------------------------------
    psi1_k = []  # Store Fourier-space first-order displacement for each axis
    disp1 = []   # Real-space first-order displacement fields
    for i, xi in enumerate(('x', 'y', 'z')):
        psi1_ki = np.zeros_like(kmod, dtype=complex)
        psi1_ki[mask] = -1j * kvec[i, mask] / (kmod[mask]**2) * delta_k[mask]
        psi1_k.append(psi1_ki)
        disp_field = np.fft.irfftn(psi1_ki, s=nvox, axes=axes)
        max_disp = np.max(np.abs(disp_field))
        log.info(f"Maximal '{xi}' 1st-order displacement: {max_disp*1e3:.3f} kpc/h; "
                 f"in units of mean particle separation: {max_disp * dk:.3f}")
        disp1.append(disp_field)
    disp1 = np.array(disp1)  # Shape: (3, nmesh, nmesh, nmesh)

    # ------------------------------
    # 2. Compute derivatives of Psi^(1) for the second-order source
    # ------------------------------
    # The derivative of the i-th component of the first-order displacement
    # Psi^(1) with respect to the j-th coordinate in Fourier space is given by
    #
    #     d[Psi^(1)_i]/dx_j = irfftn(1j * kvec[j] * psi1_k[i])
    #
    dPxx = np.fft.irfftn(1j * kvec[0] * psi1_k[0], s=nvox, axes=axes)  # d(Psi_x)/dx
    dPxy = np.fft.irfftn(1j * kvec[1] * psi1_k[0], s=nvox, axes=axes)  # d(Psi_x)/dy
    dPxz = np.fft.irfftn(1j * kvec[2] * psi1_k[0], s=nvox, axes=axes)  # d(Psi_x)/dz
    # --
    dPyy = np.fft.irfftn(1j * kvec[1] * psi1_k[1], s=nvox, axes=axes)  # d(Psi_y)/dy
    dPyz = np.fft.irfftn(1j * kvec[2] * psi1_k[1], s=nvox, axes=axes)  # d(Psi_y)/dz
    # --
    dPzz = np.fft.irfftn(1j * kvec[2] * psi1_k[2], s=nvox, axes=axes)  # d(Psi_z)/dz

    # Compute the quadratic source S(x) and its Fourier transform S(k)
    S = dPxx * dPyy + dPxx * dPzz + dPyy * dPzz - (dPxy**2 + dPxz**2 + dPyz**2)
    S_k = np.fft.rfftn(S)

    # ------------------------------
    # 3. Second-order displacement (Psi^(2))
    # ------------------------------
    # Solve the Poisson equation in Fourier space:
    # phi2(k) = -S(k) / |k|^2
    phi2_k = np.zeros_like(S_k, dtype=complex)
    phi2_k[mask] = -S_k[mask] / (kmod[mask]**2)

    # Compute the second-order displacement field:
    # Psi2_i(k) = i * k_i * phi2(k), then inverse FFT to obtain real space.
    disp2 = []  # Real-space second-order displacement fields
    for i, xi in enumerate(('x', 'y', 'z')):
        psi2_k = np.zeros_like(S_k, dtype=complex)
        psi2_k[mask] = 1j * kvec[i, mask] * phi2_k[mask]
        disp_field = np.fft.irfftn(psi2_k, s=nvox, axes=axes)
        max_disp = np.max(np.abs(disp_field))
        log.info(f"Maximal '{xi}' 2nd-order displacement: {max_disp*1e3:.3f} kpc/h; "
                 f"in units of mean particle separation: {max_disp * dk:.3f}")
        disp2.append(disp_field)
    disp2 = np.array(disp2)  # Shape: (3, nmesh, nmesh, nmesh)

    # ------------------------------
    # 4. Interpolate and update particle positions and velocities
    # ------------------------------
    # For each spatial axis, interpolate the displacement fields (both
    # first- and second-order) from the grid to the particle positions.
    for i in range(3):
        disp1_interp = interpolate_field(x, disp1[i], dk)
        disp2_interp = interpolate_field(x, disp2[i], dk)
        xpert[:, i] = x[:, i] - D1 * disp1_interp + D2 * disp2_interp
        v[:, i] = - disp1_interp * D1 * dD1 + disp2_interp * D2 * dD2
    return xpert, v / h

In [ ]:
nmesh = 64
Lbox = [100, 100, 50]
dk, nvox = cubic_voxels(nmesh, Lbox)

field = generate_normal(size=nvox, seed=seed)
field = compute_overdensity(field)

x = generate_uniform((2000, 3), seed=seed) * np.array(Lbox)
x_pert, v_pert = lpt2(x, field, nvox, dk, D1, dD1, D2, dD2, h=H0/100, counter=False)
x_pert = np.where(periodic, np.mod(x_pert, Lbox), x_pert)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

idx = [0, 1]

ax = axes[0]
ax.scatter(*x[:, idx].T, c='k', s=4**2, ec='none', alpha=0.5)
ax.set_title('Initial positions', loc='left', fontsize=10)

ax = axes[1]
ax.scatter(*x_pert[:, idx].T, c='tab:blue', s=4**2, ec='none', alpha=0.5)
ax.set_title('Perturbed (LPT2) positions', loc='left', fontsize=10)

ax = axes[2]
disp = np.column_stack((x[:, idx].ravel(), x_pert[:, idx].ravel()))
ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
ax.set_title('Displacement field', loc='left', fontsize=10)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    disp = np.column_stack((x[:, idx].ravel(), x_pert[:, idx].ravel()))
    ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

## Comparison of models

In [ ]:
nmesh = 64
Lbox = [100, 100, 50]
dk, nvox = cubic_voxels(nmesh, Lbox)

field = generate_normal(size=nvox, seed=seed)
field = compute_overdensity(field)

x = generate_uniform((100, 3), seed=seed) * np.array(Lbox)
x_pert_lpt1, v_pert_lpt1 = lpt1(x, field, nvox, dk, D1, dD1, h=H0/100, counter=False)
x_pert_lpt2, v_pert_lpt2 = lpt2(x, field, nvox, dk, D1, dD1, D2, dD2, h=H0/100, counter=False)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

idx = [0, 1]

ax = axes[0]
ax.scatter(*x[:, idx].T, c='k', s=2**2, ec='none', alpha=0.5)
ax.scatter(*x_pert_lpt1[:, idx].T, c='tab:red', s=2**2, ec='none', alpha=0.5)
ax.set_title('LPT1 Displacement', loc='left', fontsize=10)

ax = axes[1]
ax.scatter(*x[:, idx].T, c='k', s=2**2, ec='none', alpha=0.5)
ax.scatter(*x_pert_lpt2[:, idx].T, c='tab:blue', s=2**2, ec='none', alpha=0.5)
ax.set_title('LPT2 Displacement', loc='left', fontsize=10)

ax = axes[2]
ax.scatter(*x[:, idx].T, c='k', s=2**2, ec='none', alpha=0.5)
ax.scatter(*x_pert_lpt1[:, idx].T, c='tab:red', s=2**2, ec='none', alpha=0.5)
ax.scatter(*x_pert_lpt2[:, idx].T, c='tab:blue', s=2**2, ec='none', alpha=0.5)
ax.set_title('Difference between LPT1 and LPT2', loc='left', fontsize=10)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

idx = [0, 1]

ax = axes[0]
disp = np.column_stack((x[:, idx].ravel(), x_pert_lpt1[:, idx].ravel()))
ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
ax.set_title('LPT1 Displacement', loc='left', fontsize=10)

ax = axes[1]
disp = np.column_stack((x[:, idx].ravel(), x_pert_lpt2[:, idx].ravel()))
ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
ax.set_title('LPT2 Displacement', loc='left', fontsize=10)

ax = axes[2]
disp = np.column_stack((x_pert_lpt1[:, idx].ravel(), x_pert_lpt2[:, idx].ravel()))
ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
ax.set_title('Difference between LPT1 and LPT2', loc='left', fontsize=10)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    disp = np.column_stack((x_pert_lpt1[:, idx].ravel(), x_pert_lpt2[:, idx].ravel()))
    ax.plot(*disp, lw=0.5, c='k', alpha=0.5)
    ax.set_xlim(0, Lbox[idx[0]])
    ax.set_ylim(0, Lbox[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title('Difference between LPT1 and LPT2', loc='left', fontsize=10)
plt.show()